# DDM, hands on: size and interactions of proteins from downloaded movies

This notebook repeats the analysis of the [Proteins](https://rcerbino.github.io/ddm/proteins/) page with numpy only (`pip install numpy tifffile matplotlib`), on one movie per concentration. To make them: open the page at the Student level, choose the antibody and 115 mM salt, set the highest concentration to 30 mg/mL, then for each of 4.5, 9, 15, 22.5 and 30 mg/mL set the concentration, wait for the run and press "Download the movie (TIFF)" (step 2).

The steps, at each concentration: the structure function; camera noise B from the mean count; the model d = A g(q, dt) + B fitted at each q, with g taking in the exposure time; D from the weighted mean of ln D over q. Then over the series: a weighted line D = D0 (1 + k_D c) gives D0, the hydrodynamic radius and the interaction parameter k_D; the amplitude, <A> = k c / (1 + c / c0), gives the static second virial coefficient A2 from c0 = 1 / (2 A2 M), as in Guidolin et al. (2023), eq. 5. The page adds the depth of field and a jackknife, and analyses four movies per concentration; this notebook uses one.

Tested on the five movies made as above: D at each concentration within 3 % of the page's (one movie each, within the uncertainties); D0 = 41.1 +- 1.6 um^2/s (simulated 42.7), R_h = 5.5 nm (5.3), k_D = -3.5 +- 1.5 mL/g (-4.9), A2 = (4.2 +- 1.0) x 10^-5 mol mL/g^2 (4.0 x 10^-5). The page, with four movies at each concentration and the depth of field, comes closer.

## 1. What we know about the experiment

These are inputs, not results. The bench of the page: 20x objective, ORCA camera binned 2 x 2 (four pixels added after reading, so four read noises), 500 frames/s, exposure 1.99 ms, water at 22 C. The molar mass of the antibody is needed for A2.

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt

files = {4.5: "ddm-proteins-mab-4.5mgml-128x128x512.tif", 9: "ddm-proteins-mab-9mgml-128x128x512.tif",
         15: "ddm-proteins-mab-15mgml-128x128x512.tif", 22.5: "ddm-proteins-mab-22.5mgml-128x128x512.tif",
         30: "ddm-proteins-mab-30mgml-128x128x512.tif"}     # concentration (mg/mL): file
read_noise = 1.6 * 2      # electrons per binned pixel: sqrt(4) x 1.6
Te = 0.00199              # exposure time (s)
T_C, eta = 22.0, 0.955e-3 # temperature (C) and viscosity of the buffer (Pa s)
M = 148000.0              # molar mass (g/mol)

## 2. Structure function

The frames are evenly spaced, so the delays count frames; about 40 delays, log-spaced, up to half the movie. The rings of equal q leave out the central cross.

In [ ]:
def load(path):
    with tifffile.TiffFile(path) as tif:
        fr = tif.asarray().astype(np.float64)
        xr = tif.pages[0].tags["XResolution"].value
        dt = tif.imagej_metadata["finterval"]
    return fr, xr[1] / xr[0], dt

def structure(fr, dt):
    T, n, _ = fr.shape
    F = np.fft.fft2(fr)
    lags = np.unique(np.round(np.logspace(0, np.log10(T // 2), 40)).astype(int))
    k = np.fft.fftfreq(n) * n; kx, ky = np.meshgrid(k, k)
    ring = np.rint(np.hypot(kx, ky)).astype(int); ring[(kx == 0) | (ky == 0) | (ring > n // 2)] = 0
    cnt = np.bincount(ring.ravel(), minlength=n // 2 + 1)
    dq = np.array([np.bincount(ring.ravel(), (np.abs(F[l:] - F[:-l]) ** 2).mean(axis=0).ravel() / n**2, n // 2 + 1)
                   / np.maximum(cnt, 1) for l in lags])[:, 1:]
    return lags * dt, lags, dq, cnt[1:]

## 3. The model at each q

Proteins are small and many: the image is a density field, and each Fourier mode of it fades with the rate Gamma = D q^2. The camera adds up the light over each exposure, which blurs fast patterns: g(q, dt) = S(x) - K(x) exp(-Gamma dt), with x = Gamma Te, S(x) = 2 (x - 1 + e^-x) / x^2 and K(x) = 2 (cosh x - 1) / x^2. Both go to 1 for a short exposure.

The signal is weak (A is a few % to 20 % of B), so B is not fitted but taken from the camera. For each q, D is found on a grid of ln D, with A by linear least squares at each D; each delay is weighted by the number of modes and pairs over d squared.

In [ ]:
def Sx(x): return np.where(x < 1e-4, 1 - x / 3, 2 * (x - 1 + np.exp(-x)) / np.maximum(x, 1e-12) ** 2)
def Kx(x): return np.where(x < 1e-4, 1 + x * x / 12, 2 * (np.cosh(x) - 1) / np.maximum(x, 1e-12) ** 2)
def g(D, q, tau):
    G = D * q * q; x = G * Te
    return Sx(x) - Kx(x) * np.exp(-G * tau)

def fit_q(y, w, q, tau):
    def cost(lnD):
        m = g(np.exp(lnD), q, tau); A = (w * m * y).sum() / (w * m * m).sum()
        return (w * (y - A * m) ** 2).sum(), A
    grid = np.linspace(np.log(0.1), np.log(5000), 120); c = [cost(x)[0] for x in grid]; i = int(np.argmin(c))
    fine = np.linspace(grid[max(0, i - 1)], grid[min(len(grid) - 1, i + 1)], 60)
    x = fine[int(np.argmin([cost(v)[0] for v in fine]))]
    return np.exp(x), cost(x)[1], i in (0, len(grid) - 1)

def analyse(path):
    fr, px, dt = load(path)
    tau, lags, dq, nmodes = structure(fr, dt)
    T, n = fr.shape[0], fr.shape[1]
    q = 2 * np.pi * np.arange(1, n // 2 + 1) / (n * px)
    B = 2 * (fr.mean() + read_noise ** 2)
    rows = []
    for k in range(len(q)):
        if q[k] < 0.4 or q[k] > 4.5:
            continue
        y = dq[:, k] - B; w = nmodes[k] * (T - lags) / dq[:, k] ** 2
        D, A, edge = fit_q(y, w, q[k], tau)
        G = D * q[k] ** 2
        if edge or G < 2 / tau[-1] or G * tau[0] > 3:   # the rate must be within what the movie can measure
            continue
        rows.append((q[k], D, A))
    rows = np.array(rows)
    lnD = np.log(rows[:, 1])
    keep = np.abs(lnD - np.median(lnD)) < np.log(3)      # drop the rare q far off the rest
    lnD = lnD[keep]; D = np.exp(lnD.mean()); D_err = lnD.std(ddof=1) / np.sqrt(len(lnD)) * D
    band = (rows[:, 0] >= 0.6) & (rows[:, 0] <= 2.5)       # the amplitude from the same q band as the page
    return dict(D=D, D_err=D_err, A=rows[band, 2].mean(), A_err=rows[band, 2].std(ddof=1) / np.sqrt(band.sum()), B=B, rows=rows)

## 4. Each concentration

This takes a minute or two.

In [ ]:
res = {c: analyse(p) for c, p in files.items()}
for c, r in res.items():
    print(f"{c:5} mg/mL: D = {r['D']:.2f} +- {r['D_err']:.2f} um^2/s;  <A> = {r['A']:.0f} +- {r['A_err']:.0f};  A/B = {r['A'] / r['B']:.3f}")

## 5. The series: size, k_D and A2

D = D0 (1 + k_D c): a weighted straight line. D0 gives the hydrodynamic radius by the Stokes-Einstein relation, R_h = kB T / (6 pi eta D0), and the slope over D0 gives k_D (c in g/mL, k_D in mL/g).

The amplitude grows with c, but less than in proportion when the proteins repel each other (S(0) < 1): <A> = k c / (1 + c / c0), so c / <A> = (1 + c / c0) / k is a straight line, and c0 = 1 / (2 A2 M).

In [ ]:
c = np.array(list(res)); D = np.array([res[k]["D"] for k in c]); sD = np.array([res[k]["D_err"] for k in c])
(b, a), cov = np.polyfit(c, D, 1, w=1 / sD, cov="unscaled")
kD = b / a * 1e3; kD_err = abs(kD) * np.sqrt(cov[0, 0] / b**2 + cov[1, 1] / a**2 - 2 * cov[0, 1] / (a * b))
kB = 1.380649e-23
Rh = kB * (T_C + 273.15) / (6 * np.pi * eta * a * 1e-12) * 1e9
A = np.array([res[k]["A"] for k in c]); sA = np.array([res[k]["A_err"] for k in c])
y = c / A; sy = y * sA / A
(bb, aa), cov2 = np.polyfit(c, y, 1, w=1 / sy, cov="unscaled")
c0 = aa / bb; A2 = 1 / (2 * M * c0 * 1e-3)
A2_err = A2 * np.sqrt(cov2[0, 0] / bb**2 + cov2[1, 1] / aa**2 - 2 * cov2[0, 1] / (aa * bb))
print(f"D0 = {a:.2f} +- {np.sqrt(cov[1, 1]):.2f} um^2/s;  R_h = {Rh:.2f} nm;  k_D = {kD:.1f} +- {kD_err:.1f} mL/g")
print(f"c0 = {c0:.0f} mg/mL;  A2 = {A2:.2e} +- {A2_err:.1e} mol mL/g^2")

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
ax[0].errorbar(c, D, sD, fmt="o"); cc = np.linspace(0, c.max(), 50); ax[0].plot(cc, a + b * cc)
ax[0].set_xlabel("c (mg/mL)"); ax[0].set_ylabel("D (um^2/s)"); ax[0].set_title("k_D from the slope")
ax[1].errorbar(c, y, sy, fmt="o"); ax[1].plot(cc, aa + bb * cc)
ax[1].set_xlabel("c (mg/mL)"); ax[1].set_ylabel("c / <A>"); ax[1].set_title("A2 from the slope over the intercept")
fig.tight_layout()